# Phase 3 — Step 1: Mistral Sentiment Engine

Replaces VADER (English-only, 81% Neutral) with **Mistral running locally via Ollama** — a multilingual model that reads Portuguese natively.

**Stack:** DuckDB · Polars · Ollama (Mistral)

> **Pre-requisite:** Install Ollama from [ollama.com](https://ollama.com) then run in terminal: `ollama pull mistral`

In [ ]:
import polars as pl
import duckdb
import ollama
import json
import time
import os
import matplotlib.pyplot as plt

DB_PATH  = "../data/apex.duckdb"
CSV_OUT  = "../data/enriched_customer_sentiment.csv"
MODEL    = "mistral"     # Multilingual — reads Portuguese natively
SAMPLE_N = 500           # Raise to 40977 for full dataset run
BATCH    = 10
SEED     = 42
FIGURES  = "../data/figures"
os.makedirs(FIGURES, exist_ok=True)

# ── Connection test ───────────────────────────────────────────────────────────
try:
    available = [m.model for m in ollama.list().models]
    print(f"Ollama running  — models: {available}")
    if not any(MODEL in m for m in available):
        print(f"WARNING: '{MODEL}' not found. Run:  ollama pull {MODEL}")
    else:
        print(f"Model '{MODEL}' is ready.")
except Exception as e:
    print(f"ERROR — cannot reach Ollama: {e}")
    print("Start it from the Start Menu, or run:  ollama serve")


In [ ]:
reviews = pl.read_csv("../data/olist_order_reviews_dataset.csv",
    columns=["review_id", "order_id", "review_score", "review_comment_message", "review_creation_date"])

sample = (reviews.drop_nulls(subset=["review_comment_message"])
                 .sample(n=SAMPLE_N, seed=SEED)
                 .with_row_index("row_idx"))

print(f"Reviews with text : {reviews.drop_nulls(subset=['review_comment_message']).height:,}")
print(f"Sample            : {sample.height:,}")
print()
print(sample.select(["review_score", "review_comment_message"]).head(3))


In [ ]:
SYSTEM = (
    "You are a sentiment classifier for Brazilian e-commerce reviews written in Portuguese.\n"
    "For each review, classify the sentiment as exactly one of: Positive, Neutral, or Negative.\n"
    "  Positive — customer is happy, satisfied, or recommending\n"
    "  Negative — customer is unhappy, complaining, or warning\n"
    "  Neutral  — mixed feelings, very short, or unclear text\n"
    'Respond ONLY with a JSON array — no markdown, no explanation:\n'
    '[{"id": 0, "sentiment": "Positive"}, {"id": 1, "sentiment": "Negative"}, ...]'
)

VALID = {"Positive", "Neutral", "Negative"}

def sanitise(label: str) -> str:
    """Map any unexpected Mistral output to a valid label."""
    if label in VALID:
        return label
    label_lower = label.lower()
    if any(w in label_lower for w in ["posit", "good", "great", "excel", "recommend", "bom", "otimo", "recomend"]):
        return "Positive"
    if any(w in label_lower for w in ["negat", "bad", "poor", "terrible", "worst", "ruim", "pessim", "horrible"]):
        return "Negative"
    return "Neutral"

def classify_batch(texts: list[str]) -> list[str]:
    numbered = "\n".join(f"[{i}] {t[:300]}" for i, t in enumerate(texts))
    resp = ollama.chat(
        model=MODEL,
        options={"temperature": 0},
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"Classify these reviews:\n\n{numbered}"},
        ],
    )
    raw    = resp["message"]["content"].strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    parsed = json.loads(raw)
    mapping = {r["id"]: sanitise(r["sentiment"]) for r in parsed}
    return [mapping.get(i, "Neutral") for i in range(len(texts))]


sentiments = []
is_error   = []          # True = this row was a batch failure, not a real classification
texts      = sample["review_comment_message"].to_list()
errors     = 0
t0         = time.time()

print(f"Classifying {len(texts):,} reviews with Mistral (batches of {BATCH})...")

for start in range(0, len(texts), BATCH):
    chunk = texts[start:start + BATCH]
    try:
        result = classify_batch(chunk)
        sentiments.extend(result)
        is_error.extend([False] * len(chunk))
    except Exception as e:
        print(f"  Batch {start} error: {e}")
        sentiments.extend(["Neutral"] * len(chunk))
        is_error.extend([True] * len(chunk))   # flag these — NOT real classifications
        errors += 1
    done    = min(start + BATCH, len(texts))
    elapsed = time.time() - t0
    rate    = done / elapsed if elapsed > 0 else 1
    print(f"  {done}/{len(texts)}  {rate:.1f} reviews/s", end="\r")

print(f"\nDone in {time.time()-t0:.0f}s — {errors} batch errors")

sample = sample.with_columns([
    pl.Series("sentiment", sentiments),
    pl.Series("is_error",  is_error),
])

# Quality gates
invalid = [s for s in sentiments if s not in VALID]
print(f"Invalid labels after sanitise : {len(invalid)}")
print(f"Error-filled rows (excluded)  : {sum(is_error)}")
print()

# Report on clean rows only
clean = sample.filter(pl.col("is_error") == False)
print(f"Clean classifications : {len(clean):,}")
print(clean["sentiment"].value_counts())


In [ ]:
ORDER  = ["Positive", "Neutral", "Negative"]
COLORS = {"Positive": "#2ecc71", "Neutral": "#95a5a6", "Negative": "#e74c3c"}

# ── Chart 12: Sentiment breakdown ─────────────────────────────────────────────
counts = sample["sentiment"].value_counts().sort("sentiment")
pcts   = {r["sentiment"]: r["count"] / sample.height * 100
          for r in counts.iter_rows(named=True)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vals  = [pcts.get(s, 0) for s in ORDER]
bars  = axes[0].bar(ORDER, vals, color=[COLORS[s] for s in ORDER], edgecolor="white", width=0.6)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{v:.1f}%", ha="center", fontweight="bold")
axes[0].set_title(f"Chart 12 — Mistral Sentiment Breakdown\n(model: {MODEL})", fontsize=13, fontweight="bold")
axes[0].set_ylabel("% of Reviews")
axes[0].set_ylim(0, 100)
axes[0].grid(axis="y", alpha=0.3)

# ── Chart 13: Sentiment vs star rating ────────────────────────────────────────
cross = (sample.group_by(["review_score", "sentiment"])
               .agg(pl.len().alias("n"))
               .pivot(on="sentiment", index="review_score", values="n", aggregate_function="first")
               .fill_null(0)
               .sort("review_score"))
cross_pd = cross.to_pandas().set_index("review_score")
for col in ORDER:
    if col not in cross_pd.columns:
        cross_pd[col] = 0
cross_pct = cross_pd[ORDER].div(cross_pd[ORDER].sum(axis=1), axis=0) * 100

cross_pct.plot(kind="bar", ax=axes[1], stacked=True,
               color=[COLORS[s] for s in ORDER], edgecolor="white", width=0.7)
axes[1].set_title("Chart 13 — Sentiment by Star Rating\n(Validates: 5★ should be mostly Positive)",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("% of Reviews")
axes[1].set_xticklabels(["1★","2★","3★","4★","5★"], rotation=0)
axes[1].legend(title="Sentiment", loc="upper left")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.tight_layout()
plt.savefig(f"{FIGURES}/chart12_13_mistral_sentiment.png", dpi=150)
plt.show()
print("Charts 12-13 saved.")


In [ ]:
# Save only clean rows (exclude batch errors — they are NOT real sentiment)
clean = sample.filter(pl.col("is_error") == False)

OUTPUT_COLS = ["review_id","order_id","review_score",
               "review_comment_message","review_creation_date","sentiment"]

with duckdb.connect(DB_PATH) as conn:
    conn.register("sent_df", clean.select(OUTPUT_COLS))
    conn.execute("CREATE OR REPLACE TABLE sentiment AS SELECT * FROM sent_df")

clean.select(OUTPUT_COLS).write_csv(CSV_OUT)

pos = (clean["sentiment"] == "Positive").mean() or 0.0
neg = (clean["sentiment"] == "Negative").mean() or 0.0
neu = (clean["sentiment"] == "Neutral").mean() or 0.0

print("=" * 55)
print("  MISTRAL SENTIMENT ENGINE — SUMMARY")
print("=" * 55)
print(f"  Sampled          : {sample.height:,}")
print(f"  Clean rows saved : {len(clean):,}  (batch errors excluded)")
print(f"  Model            : {MODEL}  (local, free, offline)")
print(f"  Positive         : {pos:.1%}")
print(f"  Neutral          : {neu:.1%}")
print(f"  Negative         : {neg:.1%}")
print(f"  DuckDB table     : sentiment  ({len(clean):,} rows)")
print(f"  CSV out          : {CSV_OUT}")
print("=" * 55)
